# 실습 0: Amazon VPC 및 Amazon Bedrock AgentCore Gateway 설정

이 실습에서는 AWS 계정을 [부트스트랩](https://docs.aws.amazon.com/cdk/v2/guide/bootstrapping.html)하고 이후 모든 실습에서 사용할 VPC 인프라를 배포합니다. 또한 이 Notebook에서 [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore-gateway.html)를 배포합니다. 이후 실습에서도 동일한 AgentCore Gateway를 재사용합니다. 

각 VPC는 다음 세 가지 서브넷 유형으로 생성됩니다.
- **퍼블릭 서브넷**: 아웃바운드 인터넷 액세스를 위한 NAT Gateway와 Internet Gateway를 호스팅합니다.
- **프라이빗 서브넷**(NAT 포함): 워크로드가 실행되는 곳으로, NAT를 통해 인터넷에 연결할 수 있지만 외부에서 직접 액세스할 수 없습니다.
- **격리된 서브넷**: 인터넷 액세스가 전혀 없으며, 아웃바운드 연결이 없어야 하는 리소스에 사용합니다.

![다중 계정 아키텍처](./images/multi-account.png)

## 사전 요구 사항

- **[Node.js](https://nodejs.org/en/download)** v18 이상
- **[AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html)** v2
- **[Docker](https://docs.docker.com/engine/install/)**
- **[AWS CDK CLI](https://docs.aws.amazon.com/cdk/v2/guide/getting-started.html)**
- **[TypeScript](https://www.typescriptlang.org/download/)**

## 1단계: 사전 요구 사항 확인

In [ ]:
import os
from pathlib import Path

# cdk.json을 찾아 프로젝트 루트로 이동
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!node --version
!npm --version
!aws --version
!docker --version
!cdk --version

## 2단계: 프로젝트 종속성 설치

In [ ]:
# 프로젝트 종속성 설치
!npm install

## 3단계: AWS 자격 증명 구성

아래 프로파일 이름을 사용 중인 AWS CLI 프로파일에 맞게 수정합니다. 계정 ID는 자격 증명에서 자동으로 감지됩니다.

In [ ]:
import os
import json
import subprocess

# --- 프로파일 이름 수정 ---
ACCOUNT_A_PROFILE = "default"

# 자격 증명에서 계정 ID 자동 감지
result = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--profile", ACCOUNT_A_PROFILE],
    capture_output=True,
    text=True,
)
identity = json.loads(result.stdout)
ACCOUNT_A_ID = identity["Account"]

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID
os.environ["ACCOUNT_A_PROFILE"] = ACCOUNT_A_PROFILE

# 다른 Notebook에서 사용할 수 있도록 저장
%store ACCOUNT_A_ID
%store ACCOUNT_A_PROFILE

print(f"Account A: {ACCOUNT_A_ID} (profile: {ACCOUNT_A_PROFILE})")
print(f"User:      {identity['Arn']}")

## 4단계: CDK 부트스트랩

CDK 부트스트랩은 CDK 배포에 필요한 리소스(에셋용 S3 버킷, 컨테이너 이미지용 ECR 리포지토리, IAM 역할)를 프로비저닝합니다. 계정 및 리전별로 한 번만 수행하면 됩니다.

계정 A에서 **두 리전**을 부트스트랩합니다.
- `us-west-2`(기본): 모든 실습에서 사용
- `us-east-1`: [VPC Peering 실습](../01-managed-vpc-resource/02-peering.ipynb)에 필요


In [ ]:
# us-west-2에서 계정 A 부트스트랩
!cdk bootstrap aws://{ACCOUNT_A_ID}/us-west-2 --profile {ACCOUNT_A_PROFILE}

In [ ]:
## us-east-1에서 계정 A 부트스트랩, VPC Peering 실습을 실행할 때만 주석 해제
# !cdk bootstrap aws://{ACCOUNT_A_ID}/us-east-1 --profile {ACCOUNT_A_PROFILE}

## 5단계: VPC 배포

모든 VPC에는 VPC Flow Logs가 활성화되어 있으며, 트래픽 로그를 CloudWatch로 전송합니다(보존 기간 1개월).

| 스택 | 리전 | 설명 | 필요한 실습 |
|-------|--------|-------------|-------------|
| VpcegressStack-USWest2 | us-west-2 | VPC (10.0.0.0/16) | 모든 실습 |
| VpcegressStack-USEast1 | us-east-1 | VPC (10.1.0.0/16) | [VPC Peering 실습](../01-managed-vpc-resource/02-peering.ipynb) |
| PeeringApigw-USEast1 | us-east-1 | 프라이빗 API Gateway + VPCE | [VPC Peering 실습](../01-managed-vpc-resource/02-peering.ipynb) |
| VpcPeeringStack | us-west-2 | VPC 피어링 + 라우팅 | [VPC Peering 실습](../01-managed-vpc-resource/02-peering.ipynb) |

![계정 A](./images/account-a.png)

In [ ]:
# 계정 A에 VPC 배포(us-west-2)
!cdk deploy VpcegressStack-USWest2 --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file vpc-outputs-w.json

with open("vpc-outputs-w.json") as f:
    vpc_outputs = json.load(f)

# us-west-2 VPC(기본)
usw2 = vpc_outputs["VpcegressStack-USWest2"]
VPC_USW2_ID = usw2["VpcId"]
VPC_USW2_PUBLIC_SUBNETS = usw2["PublicSubnetIds"].split(",")
VPC_USW2_PRIVATE_SUBNETS = usw2["PrivateSubnetIds"].split(",")
VPC_USW2_ISOLATED_SUBNETS = usw2["IsolatedSubnetIds"].split(",")

%store VPC_USW2_ID
%store VPC_USW2_PUBLIC_SUBNETS
%store VPC_USW2_PRIVATE_SUBNETS
%store VPC_USW2_ISOLATED_SUBNETS

print("=== us-west-2 ===")
print(f"VPC ID:           {VPC_USW2_ID}")
print(f"Public subnets:   {VPC_USW2_PUBLIC_SUBNETS}")
print(f"Private subnets:  {VPC_USW2_PRIVATE_SUBNETS}")
print(f"Isolated subnets: {VPC_USW2_ISOLATED_SUBNETS}")

In [ ]:
# # VPC Peering 실습을 실행할 때만 주석 해제
# !cdk deploy VpcegressStack-USEast1 PeeringApigw-USEast1 VpcPeeringStack \
#     --profile {ACCOUNT_A_PROFILE} \
#     --require-approval never \
#     --outputs-file peering-outputs.json

# with open("peering-outputs.json") as f:
#     peering_outputs = json.load(f)

# # us-east-1 VPC
# use1 = peering_outputs["VpcegressStack-USEast1"]
# VPC_USE1_ID = use1["VpcId"]
# VPC_USE1_PUBLIC_SUBNETS = use1["PublicSubnetIds"].split(",")
# VPC_USE1_PRIVATE_SUBNETS = use1["PrivateSubnetIds"].split(",")
# VPC_USE1_ISOLATED_SUBNETS = use1["IsolatedSubnetIds"].split(",")

# %store VPC_USE1_ID
# %store VPC_USE1_PUBLIC_SUBNETS
# %store VPC_USE1_PRIVATE_SUBNETS
# %store VPC_USE1_ISOLATED_SUBNETS

# print("=== us-east-1 VPC ===")
# print(f"VPC ID:           {VPC_USE1_ID}")
# print(f"Public subnets:   {VPC_USE1_PUBLIC_SUBNETS}")
# print(f"Private subnets:  {VPC_USE1_PRIVATE_SUBNETS}")
# print(f"Isolated subnets: {VPC_USE1_ISOLATED_SUBNETS}")

# # us-east-1의 프라이빗 API Gateway(피어링 실습용)
# apigw_e = peering_outputs["PeeringApigw-USEast1"]
# PEERING_API_ID = apigw_e["ApiId"]
# PEERING_API_KEY_ID = apigw_e["ApiKeyId"]
# PEERING_VPCE_ID = apigw_e["VpceId"]
# PEERING_VPCE_SG_ID = apigw_e["VpceSgId"]

# %store PEERING_API_ID
# %store PEERING_API_KEY_ID
# %store PEERING_VPCE_ID
# %store PEERING_VPCE_SG_ID

# print("\n=== Private API Gateway (us-east-1) ===")
# print(f"API ID:   {PEERING_API_ID}")
# print(f"VPCE ID:  {PEERING_VPCE_ID}")
# print(f"VPCE SG:  {PEERING_VPCE_SG_ID}")

# # VPC 피어링
# peering_stack = peering_outputs["VpcPeeringStack"]
# PEERING_CONNECTION_ID = "peering_stack["PeeringConnectionId"]"
# %store PEERING_CONNECTION_ID
# print(f"\n=== VPC Peering ===")
# print(f"Peering ID: {PEERING_CONNECTION_ID}")

## 6단계: Amazon Bedrock AgentCore Gateway 배포

이 스택은 다음 리소스를 배포합니다.
- MCP 프로토콜과 시맨틱 검색이 활성화된 **Amazon Bedrock AgentCore Gateway**
- 머신 간(M2M) OAuth 2.0 클라이언트 자격 증명 흐름으로 구성된 **Amazon Cognito User Pool**
- VPC Lattice ENI 프로비저닝을 위한 EC2 권한이 있는 Gateway용 **IAM 실행 역할**

스택 출력(Gateway URL, Cognito 자격 증명 등)은 저장되며 `%store`를 통해 이후 Notebook과 공유됩니다.

이 데모에서는 Amazon Cognito를 사용하여 AgentCore Gateway의 인바운드 인증을 관리하지만, 엔터프라이즈 요구 사항에 따라 OAuth 2.0을 준수하는 모든 IDP를 구성할 수 있습니다. 인바운드 구성에서는 AgentCore Gateway를 호출할 수 있는 사용자를 관리합니다. Okta, Entra ID, Auth 등의 [설정](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-idp-microsoft.html) 단계를 확인하세요.

In [ ]:
!cdk deploy SharedAgentCoreGateway --profile {ACCOUNT_A_PROFILE} --require-approval never --outputs-file gateway-outputs.json

In [ ]:
with open("gateway-outputs.json") as f:
    outputs = json.load(f)["SharedAgentCoreGateway"]

GATEWAY_ID = outputs["GatewayId"]
GATEWAY_URL = outputs["GatewayUrl"]
USER_POOL_ID = outputs["UserPoolId"]
USER_POOL_CLIENT_ID = outputs["UserPoolClientId"]
TOKEN_ENDPOINT_URL = outputs["TokenEndpointUrl"]
OAUTH_SCOPES = outputs["OAuthScopes"]

%store GATEWAY_ID
%store GATEWAY_URL
%store USER_POOL_ID
%store USER_POOL_CLIENT_ID
%store TOKEN_ENDPOINT_URL
%store OAUTH_SCOPES

print(f"Gateway ID:       {GATEWAY_ID}")
print(f"Gateway URL:      {GATEWAY_URL}")
print(f"User Pool ID:     {USER_POOL_ID}")
print(f"Client ID:        {USER_POOL_CLIENT_ID}")
print(f"Token Endpoint:   {TOKEN_ENDPOINT_URL}")
print(f"OAuth Scopes:     {OAUTH_SCOPES}")

## 7단계(선택 사항): 두 번째 계정에 VPC 배포

교차 계정 테스트에 사용할 두 번째 AWS 계정이 있다면 다음 단계를 따르세요. [교차 계정 실습](../02-self-managed-lattice/02-cross-account.ipynb)을 실행하려면 이 단계가 **필수**입니다.

| 스택 | 계정 | 리전 | CIDR |
|-------|---------|--------|------|
| VpcegressStack-USWest2-AccountB | 계정 B | us-west-2 | 10.2.0.0/16 |

![다중 계정](./images/multi-account.png)

> **중요:** 장기 액세스 키(Access Key ID / Secret Access Key)를 사용하는 것은 **AWS 모범 사례가 아닙니다**. AWS에서는 `aws sso login`을 통해 임시 자격 증명을 사용하는 [IAM Identity Center](https://docs.aws.amazon.com/singlesignon/latest/userguide/what-is.html)(이전 명칭 AWS SSO)를 권장합니다. 프로덕션 및 엔터프라이즈 환경에서는 IAM Identity Center를 사용하도록 CLI 프로파일을 구성하세요. 이 실습에서는 **빠르게 시작하기 위한 용도로만** 액세스 키를 사용합니다. 자세한 내용은 [IAM 보안 모범 사례](https://docs.aws.amazon.com/IAM/latest/UserGuide/best-practices.html)를 참조하세요.


In [ ]:
# from getpass import getpass

# ACCOUNT_B_PROFILE = "account-b"

# ACCESS_KEY = getpass("AWS Access Key ID: ")
# SECRET_KEY = getpass("AWS Secret Access Key: ")
# SESSION_TOKEN = getpass("AWS Session Token (leave empty if not using temporary credentials): ")
# REGION_B = "us-west-2"

# assert ACCESS_KEY.strip(), "Access Key ID cannot be empty"
# assert SECRET_KEY.strip(), "Secret Access Key cannot be empty"

# !aws configure set aws_access_key_id {ACCESS_KEY} --profile {ACCOUNT_B_PROFILE}
# !aws configure set aws_secret_access_key {SECRET_KEY} --profile {ACCOUNT_B_PROFILE}
# !aws configure set region {REGION_B} --profile {ACCOUNT_B_PROFILE}

# if SESSION_TOKEN.strip():
#     !aws configure set aws_session_token {SESSION_TOKEN} --profile {ACCOUNT_B_PROFILE}
#     print(f"Profile '{ACCOUNT_B_PROFILE}' configured with session token.")
# else:
#     print(f"Profile '{ACCOUNT_B_PROFILE}' configured (no session token).")

In [ ]:
# # 계정 B ID 확인 및 자동 감지
# result = subprocess.run(
#     ["aws", "sts", "get-caller-identity", "--profile", ACCOUNT_B_PROFILE],
#     capture_output=True,
#     text=True,
# )
# identity = json.loads(result.stdout)
# ACCOUNT_B_ID = identity["Account"]

# os.environ["ACCOUNT_B_ID"] = ACCOUNT_B_ID
# os.environ["ACCOUNT_B_PROFILE"] = ACCOUNT_B_PROFILE

# # 다른 Notebook에서 사용할 수 있도록 저장
# %store ACCOUNT_B_ID
# %store ACCOUNT_B_PROFILE

# print(f"Account B: {ACCOUNT_B_ID} (profile: {ACCOUNT_B_PROFILE})")
# print(f"User:      {identity['Arn']}")

In [ ]:
# # 계정 B VPC 부트스트랩 및 배포
# !cdk bootstrap aws://{ACCOUNT_B_ID}/us-west-2 --profile {ACCOUNT_B_PROFILE}
# !ACCOUNT_B_ID={ACCOUNT_B_ID} cdk deploy VpcegressStack-USWest2-AccountB \
#     --profile {ACCOUNT_B_PROFILE} \
#     --require-approval never \
#     --outputs-file vpc-outputs-account-b.json

In [ ]:
# with open("vpc-outputs-account-b.json") as f:
#     vpc_b_outputs = json.load(f)

# accb = vpc_b_outputs["VpcegressStack-USWest2-AccountB"]
# VPC_ACCB_USW2_ID = accb["VpcId"]
# VPC_ACCB_USW2_PUBLIC_SUBNETS = accb["PublicSubnetIds"].split(",")
# VPC_ACCB_USW2_PRIVATE_SUBNETS = accb["PrivateSubnetIds"].split(",")
# VPC_ACCB_USW2_ISOLATED_SUBNETS = accb["IsolatedSubnetIds"].split(",")

# %store VPC_ACCB_USW2_ID
# %store VPC_ACCB_USW2_PUBLIC_SUBNETS
# %store VPC_ACCB_USW2_PRIVATE_SUBNETS
# %store VPC_ACCB_USW2_ISOLATED_SUBNETS

# print("=== Account B / us-west-2 ===")
# print(f"VPC ID:           {VPC_ACCB_USW2_ID}")
# print(f"Private subnets:  {VPC_ACCB_USW2_PRIVATE_SUBNETS}")

## 정리

이 실습에서 생성한 모든 리소스를 삭제하려면 다음 셀을 실행합니다. **다른 모든 실습을 완료한 후** 실행하세요.

> **경고:** AgentCore Gateway에 대상이 있으면 삭제할 수 없습니다. 아래 정리 셀은 먼저 모든 Gateway 대상을 자동으로 삭제한 다음 스택을 삭제합니다.

> **참고:** 다른 실습의 보안 그룹 또는 ENI가 VPC에 남아 있으면 VPC 스택(`VpcegressStack-USWest2`) 삭제에 실패할 수 있습니다. VPC Lattice Resource Gateway ENI가 여전히 해당 리소스를 참조할 수 있으므로 실습 스택을 삭제할 때도 유지됩니다. VPC 스택을 삭제하기 전에 모든 실습 스택이 삭제되었는지 확인하고, 유지된 보안 그룹과 연결이 끊긴 ENI를 AWS Console 또는 CLI를 통해 VPC에서 수동으로 삭제하세요.

In [ ]:
# import time
# import boto3

# session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name="us-west-2")
# agentcore = session.client("bedrock-agentcore-control")

# # Gateway 스택을 삭제하기 전에 모든 Gateway 대상 삭제
# targets = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID, maxResults=100)
# for target in targets.get("items", []):
#     tid = target["targetId"]
#     print(f"Deleting target: {tid}")
#     agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=tid)
#     # 삭제 완료 대기
#     while True:
#         try:
#             t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=tid)
#             print(f"  {tid}: {t['status']}")
#             time.sleep(15)
#         except agentcore.exceptions.ResourceNotFoundException:
#             print(f"  {tid}: deleted")
#             break

# print("All targets deleted. Safe to destroy gateway stack.")

In [ ]:
# # 계정 A 스택 삭제 - us-west-2
# !cdk destroy SharedAgentCoreGateway VpcegressStack-USWest2 --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # 계정 A 스택 삭제 - us-east-1(VPC Peering 실습)
# !cdk destroy VpcPeeringStack PeeringApigw-USEast1 VpcegressStack-USEast1 --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # (선택 사항) 계정 B 스택 삭제
# # !ACCOUNT_B_ID={ACCOUNT_B_ID} cdk destroy CrossAccountApigw-AccountB VpcegressStack-USWest2-AccountB --profile {ACCOUNT_B_PROFILE} --force